# Cyberbullying & Online Harassment Forensic Analyzer

**Revised notebook:** fixes ZIP dataset loading, prevents TF-IDF train/test leakage, and strengthens evidence-record integrity verification. Run all cells from a fresh Colab runtime.

In [ ]:
# Phase 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
# Upload and extract the dataset ZIP or select train.csv directly
from google.colab import files
from pathlib import Path
import zipfile
import os

uploaded = files.upload()
DATA_DIR = Path('/content/cyberbullying_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

for filename in uploaded:
    file_path = Path(filename)
    if zipfile.is_zipfile(file_path):
        extract_dir = DATA_DIR / file_path.stem
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(file_path, 'r') as archive:
            archive.extractall(extract_dir)

train_csv_candidates = list(DATA_DIR.rglob('train.csv'))
if Path('train.csv').exists():
    train_csv_candidates.append(Path('train.csv'))

if not train_csv_candidates:
    raise FileNotFoundError(
        'train.csv was not found. Upload the Jigsaw dataset ZIP or train.csv.'
    )

# Select the largest matching train.csv when multiple archives are uploaded
train_csv_path = max(train_csv_candidates, key=lambda path: path.stat().st_size)
print('Dataset file selected:', train_csv_path)


In [ ]:
# Load and validate the Jigsaw training dataset
df = pd.read_csv(train_csv_path)

required_columns = [
    'id', 'comment_text', 'toxic', 'severe_toxic',
    'obscene', 'threat', 'insult', 'identity_hate'
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required dataset columns: {missing_columns}')

print('Dataset loaded successfully!')
print('Shape:', df.shape)


In [ ]:
df.head()

In [ ]:
#Columns check
print(df.columns.tolist())

In [ ]:
#dataset information
df.info()

In [ ]:
#missing value
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Total comments:", len(df))
print("Total columns:", len(df.columns))

In [ ]:
# Count the number of comments belonging to each toxicity category

labels = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

label_counts = df[labels].sum().sort_values(ascending=False)

print(label_counts)

In [ ]:
# Create a bar chart to visualize the distribution of toxicity categories

plt.figure(figsize=(10, 5))

plt.bar(label_counts.index, label_counts.values)

plt.title("Distribution of Toxicity Categories")
plt.xlabel("Toxicity Category")
plt.ylabel("Number of Comments")
plt.xticks(rotation=30)

plt.show()

In [ ]:
# Calculate how many toxicity labels are assigned to each comment

df['label_count'] = df[labels].sum(axis=1)

print(df['label_count'].value_counts().sort_index())

In [ ]:
# Visualize how many comments have one or more toxicity labels

label_count_distribution = df['label_count'].value_counts().sort_index()

plt.figure(figsize=(8, 5))

plt.bar(
    label_count_distribution.index.astype(str),
    label_count_distribution.values
)

plt.title("Number of Toxicity Labels per Comment")
plt.xlabel("Number of Labels")
plt.ylabel("Number of Comments")

plt.show()

In [ ]:
# Calculate the number of characters in each comment

df['text_length'] = df['comment_text'].fillna('').str.len()
df[['comment_text', 'text_length']].head()



In [ ]:
# Calculate the number of words in each comment

df['word_count'] = df['comment_text'].fillna('').str.split().str.len()
df[['comment_text', 'word_count']].head()

In [ ]:
# Compare Comment Length and Word Count distributions
# Comment Length is measured in characters
# Word Count is measured in number of words

plt.figure(figsize=(14, 5))

# Comment Length Distribution
plt.subplot(1, 2, 1)
plt.hist(df['text_length'], bins=50)
plt.title("Comment Length Distribution")
plt.xlabel("Number of Characters")
plt.ylabel("Number of Comments")

# Word Count Distribution
plt.subplot(1, 2, 2)
plt.hist(df['word_count'], bins=50)
plt.title("Word Count Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Number of Comments")
plt.tight_layout()
plt.show()

In [ ]:
# Import libraries required for text cleaning and word frequency analysis
import re
from collections import Counter

# Combine all comments into one text
all_text = " ".join(df['comment_text'].fillna('').astype(str))

# Convert text to lowercase
all_text = all_text.lower()

# Keep only alphabetic words
words = re.findall(r'\b[a-z]+\b', all_text)
print("Total words found:", len(words))

In [ ]:
# Define common English stopwords to remove from the analysis

stop_words = {
    'the', 'and', 'to', 'of', 'a', 'in', 'is', 'it', 'you',
    'that', 'for', 'on', 'this', 'with', 'as', 'are', 'was',
    'be', 'have', 'has', 'i', 'he', 'she', 'they', 'we',
    'my', 'your', 'me', 'but', 'or', 'an', 'at', 'from',
    'not', 'so', 'if', 'can', 'will', 'do', 'did', 'just',
    'what', 'who', 'all', 'about', 'there', 'their'
}

# Remove stopwords
filtered_words = [word for word in words if word not in stop_words]

print("Words after stopword removal:", len(filtered_words))

In [ ]:
# Count the frequency of each word
word_frequency = Counter(filtered_words)

# Display the 20 most frequent words
top_words = word_frequency.most_common(20)
print("Top 20 Most Frequent Words:")
for word, count in top_words:
    print(word, ":", count)

In [ ]:
# Visualize the 20 most frequent words in the dataset
words_plot = [item[0] for item in top_words]
counts_plot = [item[1] for item in top_words]
plt.figure(figsize=(12, 6))
plt.bar(words_plot, counts_plot)
plt.title("Top 20 Most Frequent Words")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Phase 2 — NLP + Feature Engineering

In [ ]:
# Import libraries
from sklearn.feature_extraction.text import TfidfVectorizer
print("TF-IDF library imported successfully!")

In [ ]:
# TF-IDF features will be fitted after the train-test split to prevent data leakage
from sklearn.feature_extraction.text import TfidfVectorizer
print('TF-IDF library imported successfully!')


In [ ]:
# Prepare the toxicity labels for multi-label classification
y = df[labels]
print("Target labels prepared successfully!")
print("Target Shape:", y.shape)
# Display the first 5 target records
y.head()

In [ ]:
# Split raw comments first, then fit TF-IDF only on training comments
from sklearn.model_selection import train_test_split

X_text = df['comment_text'].fillna('').astype(str)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2)
)
X_train = tfidf_vectorizer.fit_transform(X_train_text)
X_test = tfidf_vectorizer.transform(X_test_text)

print('Train-test split and TF-IDF transformation completed!')
print('Training features:', X_train.shape)
print('Testing features:', X_test.shape)
print('Training labels:', y_train.shape)
print('Testing labels:', y_test.shape)


In [ ]:
#Phase 3 — AI Detection / Machine Learning

In [ ]:
# Train a Logistic Regression model for multi-label toxicity classification
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
# Create the Logistic Regression model
logistic_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    )
)
# Train the model
logistic_model.fit(X_train, y_train)
print("Logistic Regression model trained successfully!")

In [ ]:
# Generate toxicity predictions using the trained Logistic Regression model
y_pred = logistic_model.predict(X_test)
print("Predictions generated successfully!")
print("Prediction Shape:", y_pred.shape)
# Display predictions for the first 5 comments
print("\nFirst 5 Predictions:")
print(y_pred[:5])

In [ ]:
# Evaluate the Logistic Regression model using Precision, Recall and F1-score
from sklearn.metrics import classification_report
print("Logistic Regression Classification Report:\n")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=labels,
        zero_division=0
    )
)

In [ ]:
# Visualize Precision, Recall and F1-score for each toxicity category

from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(
    y_test, y_pred, average=None, zero_division=0
)
recall = recall_score(
    y_test, y_pred, average=None, zero_division=0
)
f1 = f1_score(
    y_test, y_pred, average=None, zero_division=0
)
x = np.arange(len(labels))
width = 0.25

plt.figure(figsize=(12, 6))
plt.bar(x - width, precision, width, label='Precision')
plt.bar(x, recall, width, label='Recall')
plt.bar(x + width, f1, width, label='F1-score')
plt.title("Logistic Regression Performance by Toxicity Category")
plt.xlabel("Toxicity Category")
plt.ylabel("Score")
plt.xticks(x, labels, rotation=30)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Create a confusion matrix for the Toxic category
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Select the Toxic category
toxic_index = labels.index('toxic')
# Calculate confusion matrix
cm = confusion_matrix(
    y_test.iloc[:, toxic_index],
    y_pred[:, toxic_index]
)
print("Confusion Matrix for Toxic Category:")
print(cm)

# Visualize the confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Toxic', 'Toxic']
)

disp.plot()
plt.title("Confusion Matrix - Toxic Category")
plt.show()

In [ ]:
# Create confusion matrices for all six toxicity categories

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, label in enumerate(labels):

    # Calculate confusion matrix for the current category
    cm = confusion_matrix(
        y_test.iloc[:, i],
        y_pred[:, i]
    )

    # Display the confusion matrix
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Negative', 'Positive']
    )

    disp.plot(
        ax=axes[i // 3, i % 3],
        colorbar=False
    )

    axes[i // 3, i % 3].set_title(label.replace('_', ' ').title())

plt.suptitle(
    "Confusion Matrices for Toxicity Categories",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate overall Precision, Recall and F1-score for the Logistic Regression model
overall_precision = precision_score(
    y_test,
    y_pred,
    average='micro',
    zero_division=0
)
overall_recall = recall_score(
    y_test,
    y_pred,
    average='micro',
    zero_division=0
)
overall_f1 = f1_score(
    y_test,
    y_pred,
    average='micro',
    zero_division=0
)
print("Logistic Regression Overall Performance")
print("---------------------------------------")
print("Precision:", round(overall_precision, 4))
print("Recall   :", round(overall_recall, 4))
print("F1-score :", round(overall_f1, 4))

In [ ]:
# Train a Linear SVM model for multi-label toxicity classification
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
# Create the Linear SVM model
svm_model = OneVsRestClassifier(
    LinearSVC(
        class_weight='balanced',
        random_state=42
    )
)
# Train the model
svm_model.fit(X_train, y_train)
print("Linear SVM model trained successfully!")

In [ ]:
# Generate toxicity predictions using the trained Linear SVM model
svm_pred = svm_model.predict(X_test)
print("Linear SVM predictions generated successfully!")
print("Prediction Shape:", svm_pred.shape)
# Display predictions for the first 5 comments
print("\nFirst 5 Predictions:")
print(svm_pred[:5])

In [ ]:
# Evaluate the Linear SVM model using Precision, Recall and F1-score
print("Linear SVM Classification Report:\n")
print(
    classification_report(
        y_test,
        svm_pred,
        target_names=labels,
        zero_division=0
    )
)

In [ ]:
# Compare Logistic Regression and Linear SVM using overall Precision, Recall and F1-score
lr_precision = precision_score(
    y_test, y_pred, average='micro', zero_division=0
)
lr_recall = recall_score(
    y_test, y_pred, average='micro', zero_division=0
)
lr_f1 = f1_score(
    y_test, y_pred, average='micro', zero_division=0
)
svm_precision = precision_score(
    y_test, svm_pred, average='micro', zero_division=0
)
svm_recall = recall_score(
    y_test, svm_pred, average='micro', zero_division=0
)
svm_f1 = f1_score(
    y_test, svm_pred, average='micro', zero_division=0
)
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Linear SVM'],
    'Precision': [lr_precision, svm_precision],
    'Recall': [lr_recall, svm_recall],
    'F1-Score': [lr_f1, svm_f1]
})
print("Model Performance Comparison:")
print(comparison.round(4))

In [ ]:
# Visualize the performance comparison between Logistic Regression and Linear SVM
comparison_plot = comparison.set_index('Model')
comparison_plot.plot(
    kind='bar',
    figsize=(10, 6)
)
plt.title("Logistic Regression vs Linear SVM")
plt.xlabel("Machine Learning Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Evaluation Metric")
plt.tight_layout()
plt.show()

In [ ]:
#PHASE 4 START — ADVANCED DEEP LEARNING

In [ ]:
# Import libraries required for the BiLSTM deep learning model
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

print("TensorFlow and Deep Learning libraries imported successfully!")
print("TensorFlow version:", tf.__version__)

In [ ]:
# Convert text comments into numerical token sequences for the BiLSTM model
tokenizer = Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)
# Learn the vocabulary from the training comments
tokenizer.fit_on_texts(
    df.loc[y_train.index, 'comment_text'].fillna('').astype(str)
)
# Convert training and testing comments into sequences
X_train_seq = tokenizer.texts_to_sequences(
    df.loc[y_train.index, 'comment_text'].fillna('').astype(str)
)
X_test_seq = tokenizer.texts_to_sequences(
    df.loc[y_test.index, 'comment_text'].fillna('').astype(str)
)
print("Text tokenization completed successfully!")
print("Training sequences:", len(X_train_seq))
print("Testing sequences:", len(X_test_seq))
print("Vocabulary size:", min(len(tokenizer.word_index) + 1, 20000))

In [ ]:
# Pad all text sequences to the same length for the BiLSTM model
MAX_LENGTH = 200
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)
print("Sequence padding completed successfully!")
print("Training padded shape:", X_train_pad.shape)
print("Testing padded shape:", X_test_pad.shape)

In [ ]:
# Rebuild the BiLSTM model using the current Keras input format
from tensorflow.keras import Input
VOCAB_SIZE = 20000
EMBEDDING_DIM = 128
OUTPUT_CLASSES = 6
bilstm_model = Sequential([

    # Define the input shape explicitly
    Input(shape=(MAX_LENGTH,)),

    # Convert word indices into dense word embeddings
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    # Learn text patterns from both forward and backward directions
    Bidirectional(
        LSTM(64, return_sequences=False)
    ),
    # Reduce overfitting
    Dropout(0.5),
    # Fully connected layer
    Dense(64, activation='relu'),
    # Six outputs for the six toxicity categories
    Dense(OUTPUT_CLASSES, activation='sigmoid')
])
print("BiLSTM model rebuilt successfully!")
bilstm_model.summary()

In [ ]:
# Compile the BiLSTM model for multi-label toxicity classification
bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='binary_accuracy')
    ]
)
print("BiLSTM model compiled successfully!")

In [ ]:
# Train the BiLSTM model on the toxicity training data
history = bilstm_model.fit(
    X_train_pad,
    y_train.values,
    validation_split=0.10,
    epochs=3,
    batch_size=128,
    verbose=1
)
print("BiLSTM training completed successfully!")

In [ ]:
# Generate toxicity predictions using the trained BiLSTM model
bilstm_probabilities = bilstm_model.predict(
    X_test_pad,
    batch_size=128,
    verbose=1
)
# Convert probabilities into binary predictions
bilstm_pred = (bilstm_probabilities >= 0.5).astype(int)
print("BiLSTM predictions generated successfully!")
print("Prediction Shape:", bilstm_pred.shape)
# Display predictions for the first 5 comments
print("\nFirst 5 Predictions:")
print(bilstm_pred[:5])

In [ ]:
# Evaluate the BiLSTM model using Precision, Recall and F1-score
bilstm_precision = precision_score(
    y_test,
    bilstm_pred,
    average='micro',
    zero_division=0
)
bilstm_recall = recall_score(
    y_test,
    bilstm_pred,
    average='micro',
    zero_division=0
)
bilstm_f1 = f1_score(
    y_test,
    bilstm_pred,
    average='micro',
    zero_division=0
)
print("BiLSTM Overall Performance")
print("--------------------------")
print("Precision:", round(bilstm_precision, 4))
print("Recall   :", round(bilstm_recall, 4))
print("F1-score :", round(bilstm_f1, 4))

In [ ]:
# Compare Logistic Regression, Linear SVM and BiLSTM performance
model_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Linear SVM',
        'BiLSTM'
    ],
    'Precision': [
        lr_precision,
        svm_precision,
        bilstm_precision
    ],
    'Recall': [
        lr_recall,
        svm_recall,
        bilstm_recall
    ],
    'F1-Score': [
        lr_f1,
        svm_f1,
        bilstm_f1
    ]
})
print("Complete Model Comparison:")
print(model_comparison.round(4))

In [ ]:
# Visualize Precision, Recall and F1-score for all three models
comparison_plot = model_comparison.set_index('Model')
comparison_plot.plot(
    kind='bar',
    figsize=(11, 6)
)
plt.title("Comparison of Machine Learning and Deep Learning Models")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Evaluation Metric")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize BiLSTM training and validation loss across epochs
plt.figure(figsize=(10, 6))
plt.plot(
    history.history['loss'],
    marker='o',
    label='Training Loss'
)
plt.plot(
    history.history['val_loss'],
    marker='o',
    label='Validation Loss'
)
plt.title("BiLSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PHASE 5 START — DIGITAL FORENSICS

In [ ]:
# Create a forensic evidence dataframe from the test comments and AI predictions
forensic_df = df.loc[y_test.index, ['comment_text']].copy()

# Add a unique evidence ID
forensic_df['evidence_id'] = [
    f"EVD-{i:05d}" for i in range(1, len(forensic_df) + 1)
]
# Store the six toxicity predictions
for i, label in enumerate(labels):
    forensic_df[label] = bilstm_pred[:, i]
# Calculate the total number of detected toxicity categories
forensic_df['toxicity_count'] = forensic_df[labels].sum(axis=1)
print("Forensic evidence dataframe created successfully!")
print("Total evidence records:", len(forensic_df))
forensic_df.head()

In [ ]:
# Assign a project-defined severity level based on the number of detected toxicity categories
def assign_severity(toxicity_count):
    if toxicity_count == 0:
        return "Non-Toxic"
    elif toxicity_count == 1:
        return "Low"
    elif toxicity_count <= 3:
        return "Medium"
    else:
        return "High"

forensic_df['severity'] = forensic_df['toxicity_count'].apply(
    assign_severity
)
print("Severity classification completed successfully!")
print("\nSeverity Distribution:")
print(forensic_df['severity'].value_counts())
print("\nSample Forensic Records:")
forensic_df[['evidence_id', 'toxicity_count', 'severity']].head(10)

In [ ]:
# Generate SHA-256 hashes for complete structured evidence records
import hashlib
import json

def generate_evidence_hash(record):
    payload = {
        'evidence_id': str(record['evidence_id']),
        'comment_text': str(record['comment_text']),
        'toxicity_labels': {label: int(record[label]) for label in labels},
        'toxicity_count': int(record['toxicity_count']),
        'severity': str(record['severity'])
    }
    canonical_record = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':')
    )
    return hashlib.sha256(canonical_record.encode('utf-8')).hexdigest()

forensic_df['evidence_hash'] = forensic_df.apply(
    generate_evidence_hash,
    axis=1
)

print('SHA-256 evidence-record hashing completed!')
print('\nSample Evidence Hashes:')
print(forensic_df[['evidence_id', 'severity', 'evidence_hash']].head(10))


In [ ]:
# Verify the integrity of a complete forensic evidence record
evidence_index = forensic_df.index[0]
evidence_record = forensic_df.loc[evidence_index]
stored_hash = evidence_record['evidence_hash']
recalculated_hash = generate_evidence_hash(evidence_record)

if stored_hash == recalculated_hash:
    integrity_status = 'VERIFIED - Evidence record unchanged'
else:
    integrity_status = 'WARNING - Evidence record may have been modified'

print('Evidence ID:', evidence_record['evidence_id'])
print('Stored Hash:     ', stored_hash)
print('Recalculated Hash:', recalculated_hash)
print('\nIntegrity Status:', integrity_status)


In [ ]:
# Filter and search forensic evidence records based on severity and toxicity categories
# Select potentially suspicious evidence
suspicious_evidence = forensic_df[
    forensic_df['severity'].isin(['High', 'Medium', 'Low'])
].copy()
print("Suspicious evidence records:", len(suspicious_evidence))

print("\nSeverity-wise Evidence Count:")
print(
    suspicious_evidence['severity']
    .value_counts()
)
# Display high-severity evidence
high_severity_evidence = forensic_df[
    forensic_df['severity'] == 'High'
]

print("\nHigh-Severity Evidence:")
print(
    high_severity_evidence[
        ['evidence_id', 'toxicity_count', 'severity', 'comment_text']
    ].head(10)
)

In [ ]:
# Analyze and visualize the number of detected records for each toxicity category\
# Count positive predictions for each toxicity category
toxicity_counts = forensic_df[labels].sum().sort_values(ascending=False)
print("Toxicity Category Counts:")
print(toxicity_counts)
# Create visualization
plt.figure(figsize=(11, 6))
toxicity_counts.plot(
    kind='bar'
)
plt.title("Detected Toxicity Categories in Forensic Evidence")
plt.xlabel("Toxicity Category")
plt.ylabel("Number of Detected Records")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the distribution of forensic evidence by severity level
severity_counts = forensic_df['severity'].value_counts()
print("Severity Distribution:")
print(severity_counts)
plt.figure(figsize=(9, 6))
severity_counts.plot(
    kind='bar'
)
plt.title("Forensic Evidence Severity Distribution")
plt.xlabel("Severity Level")
plt.ylabel("Number of Evidence Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze the relationship between different toxicity categories
toxicity_correlation = forensic_df[labels].corr()
print("Toxicity Category Correlation Matrix:")
print(toxicity_correlation.round(2))
plt.figure(figsize=(10, 7))

sns.heatmap(
    toxicity_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Correlation Between Toxicity Categories")
plt.tight_layout()
plt.show()

In [ ]:
# Generate key summary statistics for the forensic evidence dataset
total_evidence = len(forensic_df)
non_toxic_count = (
    forensic_df['severity'] == 'Non-Toxic'
).sum()

low_count = (
    forensic_df['severity'] == 'Low'
).sum()

medium_count = (
    forensic_df['severity'] == 'Medium'
).sum()

high_count = (
    forensic_df['severity'] == 'High'
).sum()

print("========== FORENSIC EVIDENCE SUMMARY ==========")

print("Total Evidence Records :", total_evidence)
print("Non-Toxic Records      :", non_toxic_count)
print("Low Severity Records   :", low_count)
print("Medium Severity Records:", medium_count)
print("High Severity Records  :", high_count)

print("\nDetected Toxicity Categories:")
for label in labels:
    print(
        f"{label.replace('_', ' ').title():18s}: "
        f"{int(forensic_df[label].sum())}"
    )

In [ ]:
# Export the complete forensic evidence dataframe as a CSV file
forensic_file = "forensic_evidence_records.csv"
forensic_df.to_csv(
    forensic_file,
    index=False
)
print("Forensic evidence dataset exported successfully!")
print("File name:", forensic_file)
print("Total records exported:", len(forensic_df))
print("Total columns:", len(forensic_df.columns))

In [ ]:
# Verify exported forensic evidence records and their SHA-256 hashes
verified_forensic_df = pd.read_csv('forensic_evidence_records.csv')

hash_check = verified_forensic_df.apply(
    lambda row: generate_evidence_hash(row) == row['evidence_hash'],
    axis=1
)

print('Exported file verification completed!')
print('Records found:', len(verified_forensic_df))
print('Columns found:', len(verified_forensic_df.columns))
print('Valid evidence hashes:', int(hash_check.sum()))
print('Invalid evidence hashes:', int((~hash_check).sum()))
print('All evidence records verified:', bool(hash_check.all()))
print('\nColumn Names:')
print(list(verified_forensic_df.columns))
print('\nFirst 5 Evidence Records:')
display(verified_forensic_df.head())


In [ ]:
# Generate the final summary of the digital forensic analysis
print("========== DIGITAL FORENSICS FINAL SUMMARY ==========")
print(f"Total Evidence Records: {len(forensic_df)}")
print("\nSeverity Distribution:")
print(
    forensic_df['severity']
    .value_counts()
)
print("\nToxicity Detection Summary:")

for label in labels:
    count = int(forensic_df[label].sum())
    percentage = (count / len(forensic_df)) * 100

    print(
        f"{label.replace('_', ' ').title():18s}: "
        f"{count:5d} records "
        f"({percentage:.2f}%)"
    )

print("\nHigh-Severity Evidence:", high_count)
print("Evidence Hashing: SHA-256")
print("Integrity Verification: Completed")
print("Evidence Export: Completed")
print("\nDigital Forensic Analysis Completed Successfully!")